# Combined EV + Heat Pump Optimization - Deterministic MPC

This notebook demonstrates how **Model Predictive Control (MPC)** can optimize both **EV charging** and **heat pump operation** simultaneously to minimize electricity costs while meeting both EV charging requirements and thermal comfort.

## Overview

**What this notebook does:**
- Loads plant data with thermal load and outdoor temperature profiles
- Calculates baseline costs for uncontrolled EV charging + uncontrolled heat pump operation
- Applies deterministic MPC optimization to schedule both EV charging and heat pump operation optimally
- Compares optimized vs baseline results to quantify cost savings

**Key optimization objectives:**
- Minimize total electricity costs (energy + peak charges + spot prices)
- Optimize EV charging based on dynamic power envelope constraints
- Optimize heat pump operation based on thermal load and outdoor temperature
- Coordinate both loads to shift operation to low-price periods
- Reduce peak power demand to lower capacity charges
- Utilize PV production when available
- Meet both EV charging requirements and thermal comfort requirements

**Expected outcomes:**
- Significant cost reduction through intelligent load coordination
- Lower peak power demand compared to uncontrolled operation
- Better alignment with renewable energy production
- Detailed comparison between baseline and optimized strategies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from IPython.display import display

# Add src to path
sys.path.append(str(Path("..") / "src"))

from optimization import deterministic_mpc_ev_hp, deterministic_mpc_ev, deterministic_mpc_hp
from billing import load_billing_config, calculate_monthly_bills, calculate_monthly_injection_bills
from heat_pump_load import calculate_heat_pump_load

print("Libraries imported successfully")

Libraries imported successfully


## Load Data

This section loads the plant data from `plant1.csv`, which contains the same dataset used in `01_benchmark.ipynb`.

**What you'll see:**
- **Data summary** showing:
  - Total number of records (15-minute intervals for one year)
  - Available columns (timestamp, grid consumption, prices, PV production, EV data, thermal load, outdoor temperature, etc.)
  - Date range of the dataset
  - Statistical summary of key variables:
    - Grid consumption (kWh)
    - PV production (kWh)
    - Grid injection (kWh)
    - EV charging (kWh)
    - Inflexible load (kWh)
    - Thermal load (kWh)
    - Outdoor temperature (°C)
    - Electricity prices (€/MWh)

This data provides the foundation for both baseline cost calculation and optimization.

In [2]:
# Load plant data (same as previous notebooks)
df = pd.read_csv("../data/plant1.csv", parse_dates=["timestamp"])

# Normalize column names (handles accidental leading/trailing spaces)
df.columns = df.columns.str.strip()

print(f"Data loaded: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nData summary:")
print(df[['grid_consumption', 'pv_production', 'grid_injection', 'ev', 'inflex_load', 'thermal_load', 'outdoor_temperature', 'price']].describe())

Data loaded: 35040 rows
Columns: ['timestamp', 'grid_consumption', 'price', 'pv_production', 'grid_injection', 'total_consumption', 'ev', 'inflex_load', 'grid_consumption_excl_ev', 'thermal_load', 'outdoor_temperature']

Date range: 2025-01-01 00:00:00+01:00 to 2025-12-31 23:45:00+01:00

Data summary:
       grid_consumption  pv_production  grid_injection            ev  \
count      35040.000000   35040.000000    35040.000000  35040.000000   
mean         330.554338      51.857346        0.004944      9.856753   
std          155.762869      87.539674        0.237750     20.675190   
min            0.000000       0.000000        0.000000      0.000000   
25%          184.625000       0.000000        0.000000      0.000000   
50%          372.750000       0.000000        0.000000      0.000000   
75%          456.750000      75.425000        0.000000      8.028882   
max          705.250000     424.100000       19.250000    148.400000   

        inflex_load  thermal_load  outdoor_tempe

## Calculate Baseline Costs (Uncontrolled EV + Uncontrolled HP)

This section calculates the electricity costs for the **baseline scenario**: uncontrolled EV charging + uncontrolled heat pump operation. This serves as the reference point for comparison with the optimized scenario.

**What you'll see:**
- **Monthly billing breakdown** for the baseline scenario showing:
  - Energy costs (volume × energy rate)
  - Spot price costs (variable electricity prices)
  - Peak-based costs (monthly peak power charges)
  - Injection revenue (from excess PV production)
  - Net total costs

**Baseline scenario:**
- EV charging: uncontrolled (baseline pattern from data)
- Heat pump: uncontrolled (operates based on thermal load demand without optimization)
- Access power (billing): same heuristics as notebook 03 — conservative monthly AP from no-HP grid peaks (+20 kW, 2024 spillover seed), plus worst-case HP electrical peak from max thermal load at COP(−10°C) from `hp.yaml` (no fixed AP table in `billing.yaml`).

**Expected baseline results (indicative; re-run after code changes):**
- Maximum monthly peak: ~2976 kW
- Total net cost: ~1,795,587 EUR

Compare total costs between this baseline and the optimized scenario (optimized EV + optimized HP).

In [4]:
# Load billing configuration
config_path = "../config/billing.yaml"
config = load_billing_config(config_path)

# Load HP configuration
hp_config_path = "../config/hp.yaml"

# Calculate uncontrolled heat pump load
# This uses the heat_pump_load module to calculate HP operation based on thermal load
# without any optimization (baseline case)
print("Calculating uncontrolled heat pump load...")
import tempfile
import os

# Save df to temporary CSV for uncontrolled HP calculation
df_temp = df.copy()
if 'timestamp' in df_temp.columns:
    df_temp['timestamp'] = df_temp['timestamp'].astype(str)
temp_data_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
df_temp.to_csv(temp_data_file.name, index=False)
temp_data_file.close()
temp_data_path = temp_data_file.name

temp_hp_output = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
temp_hp_output.close()
temp_hp_output_path = temp_hp_output.name

try:
    # Calculate uncontrolled HP load
    calculate_heat_pump_load(
        data_path=temp_data_path,
        config_path=hp_config_path,
        output_path=temp_hp_output_path
    )
    
    # Load uncontrolled HP data
    hp_load_df = pd.read_csv(temp_hp_output_path)
    # Parse hp_load_df timestamps if they're strings
    if hp_load_df['timestamp'].dtype == 'object':
        hp_load_df['timestamp'] = pd.to_datetime(hp_load_df['timestamp'], utc=True)
        hp_load_df['timestamp'] = hp_load_df['timestamp'].dt.tz_localize(None)
    
    # Merge HP load with main dataframe
    df_with_hp = df.copy()
    # Ensure df timestamps are also timezone-naive for consistent merging
    if df_with_hp['timestamp'].dtype != 'datetime64[ns]':
        df_with_hp['timestamp'] = pd.to_datetime(df_with_hp['timestamp'], utc=True)
    if df_with_hp['timestamp'].dt.tz is not None:
        df_with_hp['timestamp'] = df_with_hp['timestamp'].dt.tz_localize(None)
    
    # Merge on timestamp
    df_with_hp = df_with_hp.merge(hp_load_df[['timestamp', 'hp_electrical_load']], on='timestamp', how='left')
    df_with_hp['hp_electrical_load'] = df_with_hp['hp_electrical_load'].fillna(0.0)
    
    # Verify merge was successful
    hp_load_total = df_with_hp['hp_electrical_load'].sum()
    print(f"HP load merged successfully: {hp_load_total:.2f} kWh ({hp_load_total/1000:.2f} MWh)")
    
    # Recalculate grid consumption and grid injection with HP load
    # Power balance: Grid_consumption - Grid_injection = Inflex_load + EV + HP_load - PV_production
    df_with_hp['total_consumption_with_hp'] = df_with_hp['inflex_load'] + df_with_hp['ev'] + df_with_hp['hp_electrical_load']
    df_with_hp['grid_injection_with_hp'] = np.maximum(0, df_with_hp['pv_production'] - df_with_hp['total_consumption_with_hp'])
    df_with_hp['grid_consumption_with_hp'] = np.maximum(0, df_with_hp['total_consumption_with_hp'] - df_with_hp['pv_production'])
    
    # Create temporary dataframe with HP consumption for billing
    df_billing_hp = df_with_hp.copy()
    df_billing_hp['grid_consumption'] = df_billing_hp['grid_consumption_with_hp']
    df_billing_hp['grid_injection'] = df_billing_hp['grid_injection_with_hp']
    
    # Access power: same logic as notebook 03 (not billing.yaml fixed table).
    # 1) Conservative monthly AP from no-HP grid peaks in plant data (+ 20 kW, 2024 spillover seed).
    # 2) Add worst-case HP electrical peak: max(thermal) kW / COP(-10°C).

    df_ap = df.copy()
    ts_raw = df_ap["timestamp"]
    if ts_raw.dtype == "object" or isinstance(
        ts_raw.iloc[0] if len(ts_raw) else None, str
    ):
        ts_naive = ts_raw.astype(str).str.replace(r"[+-]\d{2}:\d{2}$", "", regex=True)
        ts_naive = pd.to_datetime(ts_naive, errors="coerce")
    else:
        ts_naive = pd.to_datetime(ts_raw, errors="coerce")
        if ts_naive.dt.tz is not None:
            ts_naive = ts_naive.dt.tz_localize(None)

    naive_timestamps = pd.to_datetime(
        ts_naive.dt.strftime("%Y-%m-%d %H:%M:%S"),
        format="%Y-%m-%d %H:%M:%S",
    )
    df_ap["month"] = naive_timestamps.dt.to_period("M")

    MARGIN_KW = 20.0
    BASELINE_2024_PEAK_GRID_KW = 2663.5
    months_2025 = pd.period_range("2025-01", "2025-12", freq="M")
    monthly_peak_baseline_kw = (
        (df_ap.groupby("month")["grid_consumption"].max() * 4.0)
        .reindex(months_2025)
        .fillna(0.0)
    )
    cummax_M_minus_1_kw = monthly_peak_baseline_kw.cummax().shift(1)
    cummax_M_minus_1_kw.loc[months_2025.min()] = BASELINE_2024_PEAK_GRID_KW
    cummax_M_minus_1_kw = cummax_M_minus_1_kw.fillna(BASELINE_2024_PEAK_GRID_KW)
    access_power_conservative = cummax_M_minus_1_kw + MARGIN_KW

    from heat_pump_load import load_hp_config, interpolate_cop

    hp_cfg = load_hp_config(hp_config_path)
    cop_at_minus10 = interpolate_cop(-10.0, hp_cfg["COP_data"])
    max_thermal_kwh = float(df_with_hp["thermal_load"].max())
    thermal_max_kw = max_thermal_kwh * 4.0
    hp_additional_peak_kw = thermal_max_kw / cop_at_minus10

    _ts = df_billing_hp["timestamp"]
    if _ts.dtype == "object" or isinstance(_ts.iloc[0] if len(_ts) else None, str):
        _ts_naive = _ts.astype(str).str.replace(r"[+-]\d{2}:\d{2}$", "", regex=True)
        _ts_naive = pd.to_datetime(_ts_naive, errors="coerce")
    else:
        _ts_naive = pd.to_datetime(_ts, errors="coerce")
        if _ts_naive.dt.tz is not None:
            _ts_naive = _ts_naive.dt.tz_localize(None)
    _naive_bill = pd.to_datetime(
        _ts_naive.dt.strftime("%Y-%m-%d %H:%M:%S"),
        format="%Y-%m-%d %H:%M:%S",
    )
    df_billing_hp["month"] = _naive_bill.dt.to_period("M")

    access_power_hp_monthly = access_power_conservative + hp_additional_peak_kw
    df_billing_hp["access_power_hp"] = df_billing_hp["month"].map(
        access_power_hp_monthly.to_dict()
    ).astype(float)

    print(
        f"Heuristic HP AP add-on (worst case @ -10°C): +{hp_additional_peak_kw:.2f} kW "
        f"(max thermal {max_thermal_kwh:.2f} kWh/15min → {thermal_max_kw:.1f} kW thermal / COP {cop_at_minus10:.2f})"
    )

    baseline_hp_bills = calculate_monthly_bills(df_billing_hp, config, access_power_col="access_power_hp")
    
    # Calculate injection revenue
    baseline_hp_injection = calculate_monthly_injection_bills(df_billing_hp, config)
    baseline_hp_injection_revenue = baseline_hp_injection['injection_net_revenue_eur'].sum()
    baseline_hp_net_total = baseline_hp_bills['total_cost_eur'].sum() - baseline_hp_injection_revenue
    
    # Calculate peak power
    baseline_hp_max_peak_kw = baseline_hp_bills['monthly_peak_kw'].max()
    
    print("="*80)
    print("BASELINE (Uncontrolled EV + Uncontrolled HP) COSTS")
    print("="*80)
    print(f"\nTotal Energy Cost:     {baseline_hp_bills['energy_cost_eur'].sum():>15,.2f} EUR")
    print(f"Total Spot Cost:       {baseline_hp_bills['spot_cost_eur'].sum():>15,.2f} EUR")
    print(f"Total Peak Cost:       {baseline_hp_bills['peak_based_cost_eur'].sum():>15,.2f} EUR")
    print(f"Injection Revenue:     {baseline_hp_injection_revenue:>15,.2f} EUR")
    print(f"{'-'*50}")
    print(f"NET TOTAL COST:        {baseline_hp_net_total:>15,.2f} EUR")
    print(f"\nMaximum Monthly Peak:  {baseline_hp_max_peak_kw:>15,.2f} kW")
    print("="*80)
    
finally:
    # Clean up temporary files
    if os.path.exists(temp_data_path):
        os.remove(temp_data_path)
    if os.path.exists(temp_hp_output_path):
        os.remove(temp_hp_output_path)

Calculating uncontrolled heat pump load...
Heat pump load calculated and saved to C:\Users\VanAmmeT\AppData\Local\Temp\tmpyn2mtm6h.csv
Total electrical load: 815099.69 kWh (815.10 MWh)
Average COP: 2.57
Min COP: 2.22, Max COP: 3.12
HP load merged successfully: 815099.69 kWh (815.10 MWh)
Heuristic HP AP add-on (worst case @ -10°C): +337.78 kW (max thermal 177.33 kWh/15min → 709.3 kW thermal / COP 2.10)
BASELINE (Uncontrolled EV + Uncontrolled HP) COSTS

Total Energy Cost:          401,920.93 EUR
Total Spot Cost:          1,155,476.24 EUR
Total Peak Cost:            247,818.16 EUR
Injection Revenue:                0.01 EUR
--------------------------------------------------
NET TOTAL COST:           1,805,215.32 EUR

Maximum Monthly Peak:         2,976.85 kW


## Run Optimization

This section runs the **deterministic MPC optimization** to find the optimal schedule for both EV charging and heat pump operation that minimizes total costs while meeting all requirements.

**What the optimization does:**
- **Optimization model setup:**
  - Creates a mathematical optimization model with constraints
  - Defines objective function to minimize total costs
  - Sets up constraints for:
    - EV power limits (maximum charging rate)
    - EV power envelope (available charging windows)
    - Daily energy requirements (must meet EV demand)
    - Heat pump power limits (maximum electrical and thermal capacity)
    - Buffer SOC limits (minimum and maximum state of charge)
    - Buffer state updates (thermal energy balance)
    - Terminal SOC (end-of-year buffer SOC = `soc_final` from `hp.yaml`)
    - Thermal load satisfaction (must meet demand)
    - Power balance (grid consumption = inflexible load + EV charging + HP electrical input - PV production)
    - Monthly peak power tracking
    - Access power rules

- **Optimization process:**
  - Solves the optimization problem for all 35,040 time periods (one year)
  - Determines optimal schedules considering:
    - Electricity prices (operate when prices are low)
    - PV production availability (prefer operation during solar generation)
    - Peak power reduction (avoid simultaneous high loads)
    - Daily energy requirements (ensure all EVs are fully charged)
    - Thermal load requirements (ensure all heating demand is met)
    - Buffer flexibility (store thermal energy when prices are low, use when prices are high)

**What you'll see:**
- Progress indicators showing optimization steps
- Model statistics (number of variables, constraints, time periods)
- Optimization results including:
  - Optimized EV charging schedule
  - Optimized HP electrical input schedule
  - Optimized HP thermal output
  - Buffer SOC profile
  - Optimized grid consumption
  - Optimized monthly peak powers
  - Optimized access power

The optimization typically takes several minutes to solve for a full year of data.

### Optimization Model: Objective Function and Constraints

The optimization model (from `src/optimization.py`) minimizes total cost while meeting all operational constraints:

#### Objective Function
Minimize: **Total Cost = Energy Cost + Spot Cost + Peak-Based Costs - Injection Revenue**

**Energy Cost:**
- Formula: `Σ(grid_consumption[t] × (energy_rate_eur_per_mwh/1000 + grid_losses_percentage × spot_price[t]))`
- Energy rate: ~30.4 €/MWh (fixed costs from config)
- Grid losses: 1.75% of spot price per interval

**Spot Cost:**
- Formula: `Σ(grid_consumption[t] × spot_price_eur_per_kwh[t])`
- Spot price converted from €/MWh to €/kWh

**Peak-Based Costs:**
- Access power cost: `Σ(access_power[m] × 2.9975458)` per month
- Monthly peak cost: `Σ(monthly_peak[m] × 4.2269465)` per month
- Over-usage cost: `Σ(rolling_max_exceedance[m] × 4.4963187)` per month

**Injection Revenue** (negative cost):
- Formula: `-Σ(grid_injection[t] × net_injection_price_eur_per_kwh[t])`
- Net injection price = spot_price - imbalance_cost (21.148 €/MWh)

#### Constraints

**EV Charging Constraints:**
1. **Power Conversion**: `ev_charge_power[t] = ev_charge[t] × 4` (kWh to kW)
2. **Power Envelope**: `ev_charge_power[t] ≤ ev_power_envelope[t]` (limited by actual charging pattern)
3. **Daily Energy**: `Σ(ev_charge[t] for t in day) = daily_ev_energy_demand[day]` (must meet daily demand)

**Heat Pump Constraints:**
1. **Thermal Output**: `hp_thermal_output[t] = hp_electrical_input[t] × COP[t]` (COP varies with outdoor temperature)
2. **Thermal Power Limit**: `hp_thermal_output[t] ≤ thermal_max_kw × 0.25` (kW, 15-min period)
3. **Electrical Input Limit**: `hp_electrical_input[t] ≤ thermal_max_kw / COP_min × 0.25` (kW, 15-min period)
4. **Modulating Operation**: `hp_electrical_input[t] ∈ [0, max_electrical_input]` (continuous variable, not binary)

**Buffer (Thermal Storage) Constraints:**
1. **Buffer Capacity**: `buffer_energy_capacity = buffer_size_m3 × water_density × cp × usable_delta_t / 3600` (kWh)
2. **SOC Limits**: `SOC_min ≤ SOC[t] ≤ SOC_max` (typically 0.20 - 0.95)
3. **Buffer State Update**: `SOC[t+1] = SOC[t] - thermal_load[t]/capacity + hp_thermal_output[t]/capacity - losses[t]`
4. **Terminal SOC**: `SOC[end] = SOC_final` (from `hp.yaml`, default 0.50 — same as notebook 03)
5. **Thermal Load Satisfaction**: Buffer SOC bounds ensure thermal load is always met (SOC cannot drop below minimum when load is high)

**Power Balance:**
- `grid_consumption[t] - grid_injection[t] = inflex_load[t] + ev_charge[t] + hp_electrical_input[t] - pv_production[t]`
- Net grid power = Total load - PV production

**Grid Power:**
- `grid_power[t] = (grid_consumption[t] - grid_injection[t]) × 4` (kW, positive = consumption)

**Peak Tracking:**
- `monthly_peak[m] ≥ grid_consumption[t] × 4` for all periods t in month m
- Peak is based on grid consumption (offtake), not net power

**Exceedance:**
- `exceedance[t] ≥ monthly_peak[m] - access_power[m]` for all periods t in month m
- Rolling max exceedance: `rolling_max_exceedance[m] ≥ exceedance[k]` for k in [m-11, m]

**Access Power Rules:**
- Access power can be increased at any time (monthly)
- Access power can only be reduced 12 months after an increase (lock-in period)
- Binary variables track increases and enforce lock-in constraints

#### Key Parameters (from `config/hp.yaml` and `config/billing.yaml`)
- **Heat Pump**: Thermal max capacity 1000 kW, COP varies from 2.22-3.12 based on outdoor temperature
- **Buffer**: Size 40 m³ (default), SOC limits 0.20-0.95, initial SOC 0.50
- **Energy costs**: Fixed rate ~30.4 €/MWh, grid losses 1.75%
- **Peak costs**: Access power 2.9975 €/kW/month, Monthly peak 4.227 €/kW/month, Over-usage 4.496 €/kW/month
- **Injection**: Imbalance cost 21.148 €/MWh 
- **Access power**:  (optimized variable)

In [5]:
# Run combined EV + HP optimization
# This function optimizes both EV charging and heat pump operation simultaneously
# in a single optimization problem, allowing coordination between the two loads

print("="*80)
print("Running Combined EV + HP Optimization")
print("="*80)

results_df, summary = deterministic_mpc_ev_hp(
    df=df,
    config_path=config_path,
    hp_config_path=hp_config_path,
    timestamp_col="timestamp",
    pv_col="pv_production",
    inflex_load_col="inflex_load",
    price_col="price",
    ev_col="ev",
    thermal_load_col="thermal_load",
    outdoor_temp_col="outdoor_temperature"
)

print("\n" + "="*80)
print("Optimization Complete")
print("="*80)
print(f"\nResults DataFrame shape: {results_df.shape}")
print(f"Columns: {list(results_df.columns)}")
print(f"\nSummary:")
for key, value in summary.items():
    if isinstance(value, (int, float)):
        print(f"  {key}: {value:,.2f}")
    else:
        print(f"  {key}: {value}")

Running Combined EV + HP Optimization
Deterministic MPC Optimization for Combined EV + Heat Pump System

[1/11] Loading billing configuration from: ../config/billing.yaml
   ✓ Billing configuration loaded

[2/11] Loading HP configuration from: ../config/hp.yaml
   ✓ HP configuration loaded
     - Thermal max capacity: 1000 kW
     - Buffer size: 80 m³
     - Buffer capacity: 928.89 kWh (calculated)
     - SOC limits: 0.20 - 0.95
     - Initial SOC: 0.50
     - Terminal SOC target: 0.50

[3/11] Preparing input data...
   Input DataFrame shape: (35040, 11)
   Columns: ['timestamp', 'grid_consumption', 'price', 'pv_production', 'grid_injection', 'total_consumption', 'ev', 'inflex_load', 'grid_consumption_excl_ev', 'thermal_load', 'outdoor_temperature']
   ✓ Filtered to 35040 periods (15-min intervals)
   Date range: 2025-01-01 00:00:00 to 2025-12-31 23:45:00

[4/11] Extracting input parameters...
   PV production: 1817081.40 kWh total
   Inflexible load: 13054151.51 kWh total
   EV demand

## Compare Optimized vs Baseline

This section provides a comprehensive comparison between the optimized and baseline scenarios.

**What you'll see:**

### Cost Comparison
- **Annual cost breakdown** comparing:
  - Baseline total costs (uncontrolled EV + uncontrolled HP)
  - Optimized total costs (optimized EV + optimized HP)
  - Total savings achieved
  - Savings percentage

### Monthly Peak Power Comparison
- **Side-by-side bar charts** showing:
  - Monthly peak power for baseline vs optimized
  - Peak power reduction achieved each month
  - Access power comparison (if optimized)

### Monthly Volume Comparison
- **Energy consumption comparison** showing:
  - Monthly volumes (MWh) for baseline vs optimized
  - Volume differences (may vary due to optimization strategy)

### Summary Statistics Table
- Detailed monthly comparison table with:
  - Peak power values
  - Access power values
  - Volume consumption
  - Reductions achieved

**Key insights:**
- The optimization typically achieves significant cost savings through intelligent load coordination
- Peak power reductions help lower capacity charges
- Load shifting to low-price periods reduces energy costs
- Better utilization of PV production reduces grid consumption

**Note**: After optimization, bills are recalculated using `billing.py` with the optimized `grid_consumption` and `grid_injection` values to ensure consistency.

In [ ]:
# Calculate billing for optimized scenario
print("Calculating optimized costs...")

# Prepare results_df for billing (rename spot_price to price)
results_df_billing = results_df.copy()
results_df_billing['price'] = results_df_billing['spot_price']

# Calculate monthly bills for optimized scenario
optimized_bills = calculate_monthly_bills(
    results_df_billing,
    config,
    access_power_col="access_power"
)

# Calculate injection revenue for optimized scenario
optimized_injection = calculate_monthly_injection_bills(
    results_df_billing,
    config
)

optimized_injection_revenue = optimized_injection['injection_net_revenue_eur'].sum()
optimized_net_total = optimized_bills['total_cost_eur'].sum() - optimized_injection_revenue

# Calculate peak power
optimized_max_peak_kw = optimized_bills['monthly_peak_kw'].max()

print("="*80)
print("OPTIMIZED (Optimized EV + Optimized HP) COSTS")
print("="*80)
print(f"\nTotal Energy Cost:     {optimized_bills['energy_cost_eur'].sum():>15,.2f} EUR")
print(f"Total Spot Cost:       {optimized_bills['spot_cost_eur'].sum():>15,.2f} EUR")
print(f"Total Peak Cost:       {optimized_bills['peak_based_cost_eur'].sum():>15,.2f} EUR")
print(f"Injection Revenue:     {optimized_injection_revenue:>15,.2f} EUR")
print(f"{'-'*50}")
print(f"NET TOTAL COST:        {optimized_net_total:>15,.2f} EUR")
print(f"\nMaximum Monthly Peak:  {optimized_max_peak_kw:>15,.2f} kW")
print("="*80)

# Calculate savings
total_savings = baseline_hp_net_total - optimized_net_total
savings_percent = (total_savings / baseline_hp_net_total * 100) if baseline_hp_net_total > 0 else 0

print(f"\nTotal Savings:          {total_savings:>15,.2f} EUR")
print(f"Savings Percentage:    {savings_percent:>15,.2f} %")
print("="*80)

## Monthly Cost Comparison Table

This section creates a detailed monthly comparison table showing baseline vs optimized costs and calculates the monthly delta (savings).

**What you'll see:**
- **Monthly breakdown** of all cost components:
  - Energy costs (baseline vs optimized)
  - Spot costs (baseline vs optimized)
  - Peak-based costs (baseline vs optimized)
  - Injection revenue (baseline vs optimized)
  - Net total costs (baseline vs optimized)
- **Delta (savings)** calculated for each component and month
- **Formatted table** with color coding for easy comparison

**Key insights:**
- Monthly savings vary based on:
  - Seasonal demand patterns
  - Price volatility
  - PV production availability
- Peak cost savings are typically consistent across months
- Energy and spot savings depend on optimization effectiveness

In [ ]:
# Create monthly comparison table
monthly_comparison = pd.DataFrame({
    'month': baseline_hp_bills['month'],
    'baseline_volume_mwh': baseline_hp_bills['volume_mwh'],
    'optimized_volume_mwh': optimized_bills['volume_mwh'],
    'baseline_access_power_kw': baseline_hp_bills['access_power_kw'],
    'optimized_access_power_kw': optimized_bills['access_power_kw'],
    'baseline_peak_kw': baseline_hp_bills['monthly_peak_kw'],
    'optimized_peak_kw': optimized_bills['monthly_peak_kw'],
    'baseline_energy': baseline_hp_bills['energy_cost_eur'],
    'optimized_energy': optimized_bills['energy_cost_eur'],
    'baseline_spot': baseline_hp_bills['spot_cost_eur'],
    'optimized_spot': optimized_bills['spot_cost_eur'],
    'baseline_peak': baseline_hp_bills['peak_based_cost_eur'],
    'optimized_peak': optimized_bills['peak_based_cost_eur'],
    'baseline_total': baseline_hp_bills['total_cost_eur'] - baseline_hp_injection['injection_net_revenue_eur'],
    'optimized_total': optimized_bills['total_cost_eur'] - optimized_injection['injection_net_revenue_eur'],
})

# Calculate deltas
monthly_comparison['delta_volume_mwh'] = monthly_comparison['optimized_volume_mwh'] - monthly_comparison['baseline_volume_mwh']
monthly_comparison['delta_access_power_kw'] = monthly_comparison['optimized_access_power_kw'] - monthly_comparison['baseline_access_power_kw']
monthly_comparison['delta_peak_kw'] = monthly_comparison['optimized_peak_kw'] - monthly_comparison['baseline_peak_kw']
monthly_comparison['delta_energy'] = monthly_comparison['optimized_energy'] - monthly_comparison['baseline_energy']
monthly_comparison['delta_spot'] = monthly_comparison['optimized_spot'] - monthly_comparison['baseline_spot']
monthly_comparison['delta_peak'] = monthly_comparison['optimized_peak'] - monthly_comparison['baseline_peak']
monthly_comparison['delta_total'] = monthly_comparison['optimized_total'] - monthly_comparison['baseline_total']

# Add individual peak cost components for breakdown analysis
monthly_comparison['baseline_access_cost'] = baseline_hp_bills['access_cost_eur'].values
monthly_comparison['baseline_monthly_peak_cost'] = baseline_hp_bills['monthly_peak_cost_eur'].values
monthly_comparison['baseline_over_usage_cost'] = baseline_hp_bills['over_usage_cost_eur'].values
monthly_comparison['optimized_access_cost'] = optimized_bills['access_cost_eur'].values
monthly_comparison['optimized_monthly_peak_cost'] = optimized_bills['monthly_peak_cost_eur'].values
monthly_comparison['optimized_over_usage_cost'] = optimized_bills['over_usage_cost_eur'].values

# Calculate deltas for each peak cost component
monthly_comparison['delta_access_cost'] = monthly_comparison['optimized_access_cost'] - monthly_comparison['baseline_access_cost']
monthly_comparison['delta_monthly_peak_cost'] = monthly_comparison['optimized_monthly_peak_cost'] - monthly_comparison['baseline_monthly_peak_cost']
monthly_comparison['delta_over_usage_cost'] = monthly_comparison['optimized_over_usage_cost'] - monthly_comparison['baseline_over_usage_cost']

# Create summary comparison table
comparison_data = {
    'Scenario': ['Baseline (Uncontrolled EV + HP)', 'Optimized (Optimized EV + HP)', 'Delta (Savings)'],
    'Total Volume (MWh)': [
        monthly_comparison['baseline_volume_mwh'].sum(),
        monthly_comparison['optimized_volume_mwh'].sum(),
        monthly_comparison['delta_volume_mwh'].sum()
    ],
    'Max Monthly Peak (kW)': [
        monthly_comparison['baseline_peak_kw'].max(),
        monthly_comparison['optimized_peak_kw'].max(),
        monthly_comparison['delta_peak_kw'].max()
    ],
    'Energy Cost (EUR)': [
        monthly_comparison['baseline_energy'].sum(),
        monthly_comparison['optimized_energy'].sum(),
        monthly_comparison['delta_energy'].sum()
    ],
    'Spot Cost (EUR)': [
        monthly_comparison['baseline_spot'].sum(),
        monthly_comparison['optimized_spot'].sum(),
        monthly_comparison['delta_spot'].sum()
    ],
    'Peak Cost (EUR)': [
        monthly_comparison['baseline_peak'].sum(),
        monthly_comparison['optimized_peak'].sum(),
        monthly_comparison['delta_peak'].sum()
    ],
    'Net Total Cost (EUR)': [
        monthly_comparison['baseline_total'].sum(),
        monthly_comparison['optimized_total'].sum(),
        monthly_comparison['delta_total'].sum()
    ],
}

comparison_df = pd.DataFrame(comparison_data)

print("="*80)
print("COST COMPARISON: BASELINE vs OPTIMIZED")
print("="*80)
display(comparison_df.style.format({
    'Total Volume (MWh)': '{:,.2f}',
    'Max Monthly Peak (kW)': '{:,.2f}',
    'Energy Cost (EUR)': '{:,.2f}',
    'Spot Cost (EUR)': '{:,.2f}',
    'Peak Cost (EUR)': '{:,.2f}',
    'Net Total Cost (EUR)': '{:,.2f}',
}))

print("\n" + "="*80)
print("MONTHLY COST COMPARISON")
print("="*80)
display(monthly_comparison.style.format({
    'baseline_volume_mwh': '{:,.2f}',
    'optimized_volume_mwh': '{:,.2f}',
    'delta_volume_mwh': '{:,.2f}',
    'baseline_access_power_kw': '{:,.2f}',
    'optimized_access_power_kw': '{:,.2f}',
    'delta_access_power_kw': '{:,.2f}',
    'baseline_peak_kw': '{:,.2f}',
    'optimized_peak_kw': '{:,.2f}',
    'delta_peak_kw': '{:,.2f}',
    'baseline_energy': '{:,.2f}',
    'optimized_energy': '{:,.2f}',
    'delta_energy': '{:,.2f}',
    'baseline_spot': '{:,.2f}',
    'optimized_spot': '{:,.2f}',
    'delta_spot': '{:,.2f}',
    'baseline_peak': '{:,.2f}',
    'optimized_peak': '{:,.2f}',
    'delta_peak': '{:,.2f}',
    'baseline_total': '{:,.2f}',
    'optimized_total': '{:,.2f}',
    'delta_total': '{:,.2f}',
}))

## Visualize Monthly Delta (Savings)

This section visualizes the monthly cost savings (delta) from optimization, showing how much is saved each month compared to the baseline scenario.

**What you'll see:**
- **Stacked bar chart** showing monthly cost savings breakdown:
  - Delta Energy Cost (savings from energy-based charges)
  - Delta Spot Cost (savings from spot price optimization)
  - Delta Peak Cost (savings from peak power reduction)
- **Total monthly savings** visualization showing net savings per month

**Key insights:**
- Savings vary by month depending on:
  - Seasonal patterns (heating demand in winter)
  - Price volatility (higher savings when prices are more variable)
  - PV production (more savings when solar generation is high)
- Peak cost savings are typically the largest component
- Energy and spot cost savings depend on load shifting effectiveness

In [ ]:
# Visualize monthly delta (savings) breakdown
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(monthly_comparison))
width = 0.6

# Stacked bars showing breakdown of delta cost
delta_energy_bars = ax.bar(x, monthly_comparison['delta_energy'], width,
                           label='Delta Energy', color='tab:blue', alpha=0.7)
delta_spot_bars = ax.bar(x, monthly_comparison['delta_spot'], width,
                         bottom=monthly_comparison['delta_energy'],
                         label='Delta Spot', color='tab:orange', alpha=0.7)
delta_peak_bars = ax.bar(x, monthly_comparison['delta_peak'], width,
                         bottom=monthly_comparison['delta_energy'] + monthly_comparison['delta_spot'],
                         label='Delta Peak', color='tab:green', alpha=0.7)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Cost Savings (EUR)', fontsize=12)
ax.set_title('Monthly Cost Savings Breakdown: Baseline vs Optimized (EV + HP)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(monthly_comparison['month'], rotation=45, ha='right')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels for total delta
for i in range(len(monthly_comparison)):
    total_delta = monthly_comparison['delta_total'].iloc[i]
    if abs(total_delta) > 100:  # Only label if significant
        ax.text(i, total_delta + (200 if total_delta > 0 else -200), 
                f'{total_delta:.0f}', ha='center', va='bottom' if total_delta > 0 else 'top', 
                fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualize total monthly delta (savings)
fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(x, monthly_comparison['delta_total'], width, 
              color=['green' if val < 0 else 'red' for val in monthly_comparison['delta_total']],
              alpha=0.7)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Cost Savings (EUR)', fontsize=12)
ax.set_title('Monthly Total Cost Savings: Optimized vs Baseline (EV + HP)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(monthly_comparison['month'], rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels
for bar, val in zip(bars, monthly_comparison['delta_total']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.0f}', ha='center', va='bottom' if height > 0 else 'top', 
            fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTotal annual savings: {monthly_comparison['delta_total'].sum():,.2f} EUR")
print(f"Average monthly savings: {monthly_comparison['delta_total'].mean():,.2f} EUR")

# Visualization: Delta Peak Cost Breakdown per Month
# Split delta peak cost into: delta access power cost, delta monthly peak cost, delta exceedance cost
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(monthly_comparison))
width = 0.6

# Stacked bars showing breakdown of delta peak cost
delta_access_bars = ax.bar(x, monthly_comparison['delta_access_cost'], width,
                           label='Delta Access Power Cost', color='#2E86AB', alpha=0.8)
delta_monthly_peak_bars = ax.bar(x, monthly_comparison['delta_monthly_peak_cost'], width,
                                  bottom=monthly_comparison['delta_access_cost'],
                                  label='Delta Monthly Peak Cost', color='#A23B72', alpha=0.8)
delta_exceedance_bars = ax.bar(x, monthly_comparison['delta_over_usage_cost'], width,
                                bottom=monthly_comparison['delta_access_cost'] + monthly_comparison['delta_monthly_peak_cost'],
                                label='Delta Exceedance Cost', color='#F18F01', alpha=0.8)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Peak Cost Savings (EUR)', fontsize=12)
ax.set_title('Monthly Delta Peak Cost Breakdown: Baseline vs Optimized (EV + HP)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(monthly_comparison['month'], rotation=45, ha='right')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add value labels on bars if they're significant
for i in range(len(monthly_comparison)):
    total_delta_peak = monthly_comparison['delta_peak'].iloc[i]
    if abs(total_delta_peak) > 100:  # Only label if significant
        ax.text(i, total_delta_peak + (100 if total_delta_peak > 0 else -100), 
                f'{total_delta_peak:.0f}', ha='center', va='bottom' if total_delta_peak > 0 else 'top', 
                fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary of peak cost savings breakdown
print("\n" + "="*80)
print("PEAK COST SAVINGS BREAKDOWN")
print("="*80)
print(f"Total Access Power Cost Savings:     {monthly_comparison['delta_access_cost'].sum():>15,.2f} EUR")
print(f"Total Monthly Peak Cost Savings:    {monthly_comparison['delta_monthly_peak_cost'].sum():>15,.2f} EUR")
print(f"Total Exceedance Cost Savings:      {monthly_comparison['delta_over_usage_cost'].sum():>15,.2f} EUR")
print(f"{'-'*50}")
print(f"Total Peak Cost Savings:             {monthly_comparison['delta_peak'].sum():>15,.2f} EUR")
print("="*80)

## Visualize Results

This section provides detailed visualizations comparing the optimized and baseline operation strategies for both EV charging and heat pump.

**What you'll see:**

### Weekly Comparison Plots
- **Multi-panel visualization** for a selected week (starting September 29, 2025) showing:
  1. **EV Charging Power (kW)**: 
     - Optimized charging schedule (step plot)
     - Shows how optimization shifts charging to different times
  
  2. **HP Electrical Power (kW)**:
     - Optimized HP electrical input schedule (step plot)
     - Demonstrates how HP operation is shifted to low-price periods
  
  3. **HP Thermal Power (kW)**:
     - Optimized HP thermal output (step plot)
     - Shows thermal energy production
  
  4. **Buffer SOC (%)**:
     - Thermal buffer state of charge over time
     - Shows how buffer is used to shift HP operation
  
  5. **Spot Price (EUR/MWh)**:
     - Electricity price throughout the week
     - Helps understand why loads are shifted to specific periods

### Daily Comparison Plot
- **Detailed daily view** showing:
  - Optimized EV charging schedule for a specific day
  - Optimized HP electrical and thermal power
  - Buffer SOC profile
  - Grid consumption comparison
  - PV production (if available)
  - Spot prices
  - Inflexible load

**Key observations:**
- Optimized operation typically occurs during:
  - Low-price periods (overnight, midday when prices drop)
  - High PV production periods (to use solar energy)
  - Off-peak hours (to avoid capacity charges)
- Buffer allows HP to operate when prices are low and store thermal energy
- EV and HP loads are coordinated to avoid simultaneous peaks

These visualizations clearly demonstrate how intelligent scheduling can reduce costs while meeting all operational requirements.

In [ ]:
# Weekly and Daily Operation Plots
# Select week and day to visualize

# Select week to visualize - starting from September 29, 2025
start_week = pd.Timestamp("2025-12-25 00:00:00")
end_week = start_week + pd.Timedelta(days=7)

# Calculate week number for display purposes
year_start = pd.Timestamp("2025-01-01 00:00:00")
days_from_start = (start_week - year_start).days
week_number = (days_from_start // 7) + 1

print(f"Visualizing week {week_number}: {start_week.strftime('%Y-%m-%d')} to {end_week.strftime('%Y-%m-%d')}")

# Filter data for selected week
week_data = results_df[(results_df['timestamp'] >= start_week) & (results_df['timestamp'] < end_week)].copy()

# Weekly plot: EV + HP operation over one week
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Plot 1: EV and HP Power
ax1_price = ax1.twinx()
ev_power_kw = week_data['ev_charge_power']
hp_electrical_power_kw = week_data['hp_electrical_input'] * 4  # Convert kWh to kW
ax1.step(week_data['timestamp'], ev_power_kw, label='EV Charging Power', color='blue', linewidth=2, where='post')
ax1.step(week_data['timestamp'], hp_electrical_power_kw, label='HP Electrical Power', color='red', linewidth=2, where='post')
ax1.step(week_data['timestamp'], week_data['hp_thermal_power'], label='HP Thermal Power', color='green', linewidth=2, alpha=0.7, where='post')
ax1.set_ylabel('Power (kW)', fontsize=11, fontweight='bold')
ax1.set_title(f'EV Charging and Heat Pump Power - Week {week_number} ({start_week.strftime("%Y-%m-%d")} to {end_week.strftime("%Y-%m-%d")})', 
              fontsize=13, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1_price.plot(week_data['timestamp'], week_data['spot_price'], label='Spot price', color='purple', linewidth=1, alpha=0.7)
ax1_price.set_ylabel('Spot Price (EUR/MWh)', fontsize=11, fontweight='bold', color='purple')
ax1_price.tick_params(axis='y', labelcolor='purple')
ax1_price.legend(loc='upper right')

# Plot 2: Buffer SOC
ax2.plot(week_data['timestamp'], week_data['buffer_soc'] * 100, label='Buffer SOC', color='orange', linewidth=2)
ax2.fill_between(week_data['timestamp'], week_data['buffer_soc'] * 100, alpha=0.3, color='orange')
ax2.set_ylabel('Buffer SOC (%)', fontsize=11, fontweight='bold')
ax2.set_title('Thermal Buffer State of Charge', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 100])

# Plot 3: Spot Price and Grid Power
ax3.plot(week_data['timestamp'], week_data['spot_price'], label='Spot price', color='purple', linewidth=1)
ax3.set_ylabel('Spot Price (EUR/MWh)', fontsize=11, fontweight='bold')
ax3.set_xlabel('Timestamp', fontsize=11, fontweight='bold')
ax3.set_title('Electricity Spot Price', fontsize=13, fontweight='bold')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Daily plot: Select a specific day from the selected week
day_of_week = 7  # Change this to select a day (1-7, where 1 is the first day of the selected week)

# Calculate the specific day to plot
selected_day_start = start_week + pd.Timedelta(days=day_of_week - 1)
selected_day_end = selected_day_start + pd.Timedelta(days=1)

print(f"\nVisualizing day {day_of_week} of week {week_number}: {selected_day_start.strftime('%Y-%m-%d')}")

# Filter data for selected day
day_data = results_df[(results_df['timestamp'] >= selected_day_start) & (results_df['timestamp'] < selected_day_end)].copy()

# Daily plot: Detailed view of one day
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Plot 1: EV and HP Power
ax1_price = ax1.twinx()
ev_power_kw_day = day_data['ev_charge_power']
hp_electrical_power_kw_day = day_data['hp_electrical_input'] * 4
ax1.step(day_data['timestamp'], ev_power_kw_day, label='EV Charging Power', color='blue', linewidth=2, where='post')
ax1.step(day_data['timestamp'], hp_electrical_power_kw_day, label='HP Electrical Power', color='red', linewidth=2, where='post')
ax1.step(day_data['timestamp'], day_data['hp_thermal_power'], label='HP Thermal Power', color='green', linewidth=2, alpha=0.7, where='post')
ax1.set_ylabel('Power (kW)', fontsize=11, fontweight='bold')
ax1.set_title(f'EV Charging and Heat Pump Power - {selected_day_start.strftime("%Y-%m-%d")} (Day {day_of_week} of Week {week_number})', 
              fontsize=13, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1_price.plot(day_data['timestamp'], day_data['spot_price'], label='Spot price', color='purple', linewidth=1, alpha=0.7)
ax1_price.set_ylabel('Spot Price (EUR/MWh)', fontsize=11, fontweight='bold', color='purple')
ax1_price.tick_params(axis='y', labelcolor='purple')
ax1_price.legend(loc='upper right')

# Plot 2: Buffer SOC
ax2.plot(day_data['timestamp'], day_data['buffer_soc'] * 100, label='Buffer SOC', color='orange', linewidth=2, marker='o', markersize=3)
ax2.fill_between(day_data['timestamp'], day_data['buffer_soc'] * 100, alpha=0.3, color='orange')
ax2.set_ylabel('Buffer SOC (%)', fontsize=11, fontweight='bold')
ax2.set_title('Thermal Buffer State of Charge', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 100])

# Plot 3: Spot Price and Grid Power
ax3.plot(day_data['timestamp'], day_data['spot_price'], label='Spot price', color='purple', linewidth=1, marker='o', markersize=3)
ax3.set_ylabel('Spot Price (EUR/MWh)', fontsize=11, fontweight='bold')
ax3.set_xlabel('Timestamp', fontsize=11, fontweight='bold')
ax3.set_title('Electricity Spot Price', fontsize=13, fontweight='bold')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDay summary:")
print(f"  Total EV energy: {day_data['ev_charge'].sum():.2f} kWh")
print(f"  Total HP electrical energy: {day_data['hp_electrical_input'].sum():.2f} kWh")
print(f"  Total HP thermal energy: {day_data['hp_thermal_output'].sum():.2f} kWh")
print(f"  Average buffer SOC: {day_data['buffer_soc'].mean()*100:.1f} %")
print(f"  Average spot price: {day_data['spot_price'].mean():.2f} EUR/MWh")

## Four-Case Comparison: All Scenarios

This section compares **four different scenarios** to understand the impact of optimizing EV charging and/or heat pump operation:

1. **Case 1: Uncontrolled EV + Uncontrolled HP** (baseline)
   - EV charging: uncontrolled (baseline pattern from data)
   - Heat pump: uncontrolled (operates based on thermal load demand)

2. **Case 2: Optimized HP + Uncontrolled EV**
   - EV charging: uncontrolled (baseline pattern from data)
   - Heat pump: optimized (deterministic MPC optimization)

3. **Case 3: Optimized EV + Uncontrolled HP**
   - EV charging: optimized (deterministic MPC optimization)
   - Heat pump: uncontrolled (operates based on thermal load demand)

4. **Case 4: Optimized EV + Optimized HP** (combined optimization)
   - EV charging: optimized (deterministic MPC optimization)
   - Heat pump: optimized (deterministic MPC optimization)
   - Both loads optimized simultaneously

**Note:** All cases use a **40 m³ buffer** for the heat pump system.

### Case 1: Uncontrolled EV + Uncontrolled HP (Baseline)

This case was already calculated above. We'll reuse those results.

### Case 2: Optimized HP + Uncontrolled EV

This case optimizes only the heat pump while keeping EV charging uncontrolled. We'll load the results from notebook 03 (deterministic_mpc_hp function).

In [ ]:
# Case 2: Optimized HP + Uncontrolled EV
# Run deterministic MPC optimization for HP only (EV remains uncontrolled)
print("="*80)
print("CASE 2: Optimized HP + Uncontrolled EV")
print("="*80)

# Ensure deterministic_mpc_hp is imported (in case imports cell wasn't re-run)
try:
    deterministic_mpc_hp
except NameError:
    from optimization import deterministic_mpc_hp

# Run HP optimization (EV is treated as baseline/uncontrollable)
results_df_case2, summary_case2 = deterministic_mpc_hp(
    df=df,
    config_path=config_path,
    hp_config_path=hp_config_path,
    timestamp_col="timestamp",
    pv_col="pv_production",
    inflex_load_col="inflex_load",
    price_col="price",
    ev_col="ev",
    thermal_load_col="thermal_load",
    outdoor_temp_col="outdoor_temperature"
)

# Calculate billing for Case 2
results_df_case2_billing = results_df_case2.copy()
results_df_case2_billing['price'] = results_df_case2_billing['spot_price']

case2_bills = calculate_monthly_bills(
    results_df_case2_billing,
    config,
    access_power_col="access_power"
)

case2_injection = calculate_monthly_injection_bills(
    results_df_case2_billing,
    config
)

case2_injection_revenue = case2_injection['injection_net_revenue_eur'].sum()
case2_net_total = case2_bills['total_cost_eur'].sum() - case2_injection_revenue
case2_max_peak_kw = case2_bills['monthly_peak_kw'].max()

print(f"\nCase 2 Total Cost: {case2_net_total:,.2f} EUR")
print(f"Case 2 Max Peak: {case2_max_peak_kw:,.2f} kW")

### Case 3: Optimized EV + Uncontrolled HP

This case optimizes only the EV charging while keeping the heat pump uncontrolled. We need to:
1. Load the uncontrolled HP load from `output/uncontrolled_hp.csv`
2. Add it to the inflexible load
3. Run EV-only optimization
4. Calculate billing

In [ ]:
# Case 3: Optimized EV + Uncontrolled HP
print("="*80)
print("CASE 3: Optimized EV + Uncontrolled HP")
print("="*80)

# Load uncontrolled HP load
uncontrolled_hp_path = Path("../output/uncontrolled_hp.csv")
if not uncontrolled_hp_path.exists():
    # If file doesn't exist, calculate it first
    print("Uncontrolled HP file not found. Calculating uncontrolled HP load...")
    import tempfile
    import os
    
    df_temp = df.copy()
    if 'timestamp' in df_temp.columns:
        df_temp['timestamp'] = df_temp['timestamp'].astype(str)
    temp_data_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
    df_temp.to_csv(temp_data_file.name, index=False)
    temp_data_file.close()
    temp_data_path = temp_data_file.name
    
    temp_hp_output = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
    temp_hp_output.close()
    temp_hp_output_path = temp_hp_output.name
    
    try:
        calculate_heat_pump_load(
            data_path=temp_data_path,
            config_path=hp_config_path,
            output_path=temp_hp_output_path
        )
        # Copy to output directory
        import shutil
        output_dir = Path("../output")
        output_dir.mkdir(exist_ok=True)
        shutil.copy(temp_hp_output_path, uncontrolled_hp_path)
    finally:
        if os.path.exists(temp_data_path):
            os.remove(temp_data_path)
        if os.path.exists(temp_hp_output_path):
            os.remove(temp_hp_output_path)

# Load uncontrolled HP load
hp_load_df_case3 = pd.read_csv(uncontrolled_hp_path)

# Parse timestamps
if hp_load_df_case3['timestamp'].dtype == 'object':
    hp_load_df_case3['timestamp'] = pd.to_datetime(hp_load_df_case3['timestamp'], utc=True)
    hp_load_df_case3['timestamp'] = hp_load_df_case3['timestamp'].dt.tz_localize(None)

# Prepare dataframe for EV optimization with HP load added to inflex_load
df_case3 = df.copy()

# Ensure timestamps are timezone-naive for merging
if df_case3['timestamp'].dtype != 'datetime64[ns]':
    df_case3['timestamp'] = pd.to_datetime(df_case3['timestamp'], utc=True)
if df_case3['timestamp'].dt.tz is not None:
    df_case3['timestamp'] = df_case3['timestamp'].dt.tz_localize(None)

# Merge HP load
df_case3 = df_case3.merge(hp_load_df_case3[['timestamp', 'hp_electrical_load']], on='timestamp', how='left')
df_case3['hp_electrical_load'] = df_case3['hp_electrical_load'].fillna(0.0)

# Add HP load to inflex_load for EV optimization
# The EV optimizer will see inflex_load + HP_load as the total inflexible load
df_case3['inflex_load_with_hp'] = df_case3['inflex_load'] + df_case3['hp_electrical_load']

print(f"HP load merged successfully: {df_case3['hp_electrical_load'].sum():.2f} kWh ({df_case3['hp_electrical_load'].sum()/1000:.2f} MWh)")
print(f"Total inflexible load (with HP): {df_case3['inflex_load_with_hp'].sum():.2f} kWh ({df_case3['inflex_load_with_hp'].sum()/1000:.2f} MWh)")

# Ensure deterministic_mpc_ev is imported (in case imports cell wasn't re-run)
try:
    deterministic_mpc_ev
except NameError:
    from optimization import deterministic_mpc_ev

# Run EV-only optimization with HP load included in inflex_load
results_df_case3, summary_case3 = deterministic_mpc_ev(
    df=df_case3,
    config_path=config_path,
    timestamp_col="timestamp",
    pv_col="pv_production",
    inflex_load_col="inflex_load_with_hp",  # Use inflex_load + HP_load
    price_col="price",
    ev_col="ev"
)

# Calculate billing for Case 3
# The optimization already calculated grid_consumption and grid_injection
# based on inflex_load_with_hp + ev_charge - pv_production
results_df_case3_billing = results_df_case3.copy()
results_df_case3_billing['price'] = results_df_case3_billing['spot_price']

case3_bills = calculate_monthly_bills(
    results_df_case3_billing,
    config,
    access_power_col="access_power"
)

case3_injection = calculate_monthly_injection_bills(
    results_df_case3_billing,
    config
)

case3_injection_revenue = case3_injection['injection_net_revenue_eur'].sum()
case3_net_total = case3_bills['total_cost_eur'].sum() - case3_injection_revenue
case3_max_peak_kw = case3_bills['monthly_peak_kw'].max()

print(f"\nCase 3 Total Cost: {case3_net_total:,.2f} EUR")
print(f"Case 3 Max Peak: {case3_max_peak_kw:,.2f} kW")

### Case 4: Optimized EV + Optimized HP (Combined)

This case was already calculated above. We'll reuse those results.

### Four-Case Comparison Table

This section creates a comprehensive comparison table showing all metrics for the four cases, including consumption breakdowns, peak powers, access powers, and total costs.

In [ ]:
# Prepare data for all 4 cases
# Case 1: Uncontrolled EV + Uncontrolled HP (baseline_hp)
# Case 2: Optimized HP + Uncontrolled EV (results_df_case2)
# Case 3: Optimized EV + Uncontrolled HP (results_df_case3)
# Case 4: Optimized EV + Optimized HP (results_df)

# Calculate consumption metrics for each case (in MWh)
# Case 1: Baseline (uncontrolled EV + uncontrolled HP)
case1_ev_mwh = df['ev'].sum() / 1000.0
case1_hp_mwh = df_with_hp['hp_electrical_load'].sum() / 1000.0
case1_inflex_mwh = df['inflex_load'].sum() / 1000.0
case1_pv_mwh = df['pv_production'].sum() / 1000.0
case1_total_consumption_mwh = case1_inflex_mwh + case1_ev_mwh + case1_hp_mwh
case1_grid_consumption_mwh = baseline_hp_bills['volume_mwh'].sum()
case1_grid_injection_mwh = baseline_hp_injection['injected_volume_mwh'].sum()
# Use MAX access power throughout the year (optimizer can adjust month-by-month)
case1_access_power_kw = baseline_hp_bills['access_power_kw'].max()
case1_max_peak_kw = baseline_hp_bills['monthly_peak_kw'].max()
case1_total_cost = baseline_hp_net_total
# Baseline cost components
case1_energy_cost = baseline_hp_bills['energy_cost_eur'].sum()
case1_spot_cost = baseline_hp_bills['spot_cost_eur'].sum()
case1_peak_cost_total = baseline_hp_bills['peak_based_cost_eur'].sum()
case1_access_cost = baseline_hp_bills['access_cost_eur'].sum()
case1_monthly_peak_cost = baseline_hp_bills['monthly_peak_cost_eur'].sum()
case1_over_usage_cost = baseline_hp_bills['over_usage_cost_eur'].sum()

# Case 2: Optimized HP + Uncontrolled EV
case2_ev_mwh = df['ev'].sum() / 1000.0  # Uncontrolled EV
case2_hp_mwh = results_df_case2['hp_electrical_input'].sum() / 1000.0  # Optimized HP
case2_inflex_mwh = df['inflex_load'].sum() / 1000.0
case2_pv_mwh = df['pv_production'].sum() / 1000.0
case2_total_consumption_mwh = case2_inflex_mwh + case2_ev_mwh + case2_hp_mwh
case2_grid_consumption_mwh = case2_bills['volume_mwh'].sum()
case2_grid_injection_mwh = case2_injection['injected_volume_mwh'].sum()
# Use MAX access power throughout the year (optimizer can adjust month-by-month)
case2_access_power_kw = case2_bills['access_power_kw'].max()
case2_max_peak_kw = case2_bills['monthly_peak_kw'].max()
case2_total_cost = case2_net_total
# Case 2 cost components
case2_energy_cost = case2_bills['energy_cost_eur'].sum()
case2_spot_cost = case2_bills['spot_cost_eur'].sum()
case2_peak_cost_total = case2_bills['peak_based_cost_eur'].sum()
case2_access_cost = case2_bills['access_cost_eur'].sum()
case2_monthly_peak_cost = case2_bills['monthly_peak_cost_eur'].sum()
case2_over_usage_cost = case2_bills['over_usage_cost_eur'].sum()
# Case 2 savings vs baseline
case2_energy_savings = case2_energy_cost - case1_energy_cost
case2_spot_savings = case2_spot_cost - case1_spot_cost
case2_peak_savings_total = case2_peak_cost_total - case1_peak_cost_total
case2_access_savings = case2_access_cost - case1_access_cost
case2_monthly_peak_savings = case2_monthly_peak_cost - case1_monthly_peak_cost
case2_over_usage_savings = case2_over_usage_cost - case1_over_usage_cost

# Case 3: Optimized EV + Uncontrolled HP
case3_ev_mwh = results_df_case3['ev_charge'].sum() / 1000.0  # Optimized EV
case3_hp_mwh = df_case3['hp_electrical_load'].sum() / 1000.0  # Uncontrolled HP
case3_inflex_mwh = df['inflex_load'].sum() / 1000.0
case3_pv_mwh = df['pv_production'].sum() / 1000.0
case3_total_consumption_mwh = case3_inflex_mwh + case3_ev_mwh + case3_hp_mwh
case3_grid_consumption_mwh = case3_bills['volume_mwh'].sum()
case3_grid_injection_mwh = case3_injection['injected_volume_mwh'].sum()
# Use MAX access power throughout the year (optimizer can adjust month-by-month)
case3_access_power_kw = case3_bills['access_power_kw'].max()
case3_max_peak_kw = case3_bills['monthly_peak_kw'].max()
case3_total_cost = case3_net_total
# Case 3 cost components
case3_energy_cost = case3_bills['energy_cost_eur'].sum()
case3_spot_cost = case3_bills['spot_cost_eur'].sum()
case3_peak_cost_total = case3_bills['peak_based_cost_eur'].sum()
case3_access_cost = case3_bills['access_cost_eur'].sum()
case3_monthly_peak_cost = case3_bills['monthly_peak_cost_eur'].sum()
case3_over_usage_cost = case3_bills['over_usage_cost_eur'].sum()
# Case 3 savings vs baseline
case3_energy_savings = case3_energy_cost - case1_energy_cost
case3_spot_savings = case3_spot_cost - case1_spot_cost
case3_peak_savings_total = case3_peak_cost_total - case1_peak_cost_total
case3_access_savings = case3_access_cost - case1_access_cost
case3_monthly_peak_savings = case3_monthly_peak_cost - case1_monthly_peak_cost
case3_over_usage_savings = case3_over_usage_cost - case1_over_usage_cost

# Case 4: Optimized EV + Optimized HP
case4_ev_mwh = results_df['ev_charge'].sum() / 1000.0  # Optimized EV
case4_hp_mwh = results_df['hp_electrical_input'].sum() / 1000.0  # Optimized HP
case4_inflex_mwh = df['inflex_load'].sum() / 1000.0
case4_pv_mwh = df['pv_production'].sum() / 1000.0
case4_total_consumption_mwh = case4_inflex_mwh + case4_ev_mwh + case4_hp_mwh
case4_grid_consumption_mwh = optimized_bills['volume_mwh'].sum()
case4_grid_injection_mwh = optimized_injection['injected_volume_mwh'].sum()
# Use MAX access power throughout the year (optimizer can adjust month-by-month)
case4_access_power_kw = optimized_bills['access_power_kw'].max()
case4_max_peak_kw = optimized_bills['monthly_peak_kw'].max()
case4_total_cost = optimized_net_total
# Case 4 cost components
case4_energy_cost = optimized_bills['energy_cost_eur'].sum()
case4_spot_cost = optimized_bills['spot_cost_eur'].sum()
case4_peak_cost_total = optimized_bills['peak_based_cost_eur'].sum()
case4_access_cost = optimized_bills['access_cost_eur'].sum()
case4_monthly_peak_cost = optimized_bills['monthly_peak_cost_eur'].sum()
case4_over_usage_cost = optimized_bills['over_usage_cost_eur'].sum()
# Case 4 savings vs baseline
case4_energy_savings = case4_energy_cost - case1_energy_cost
case4_spot_savings = case4_spot_cost - case1_spot_cost
case4_peak_savings_total = case4_peak_cost_total - case1_peak_cost_total
case4_access_savings = case4_access_cost - case1_access_cost
case4_monthly_peak_savings = case4_monthly_peak_cost - case1_monthly_peak_cost
case4_over_usage_savings = case4_over_usage_cost - case1_over_usage_cost

# Create comparison table
comparison_table = pd.DataFrame({
    'Case': [
        'Case 1: Uncontrolled EV + Uncontrolled HP',
        'Case 2: Optimized HP + Uncontrolled EV',
        'Case 3: Optimized EV + Uncontrolled HP',
        'Case 4: Optimized EV + Optimized HP'
    ],
    'Total Consumption (MWh)': [
        case1_total_consumption_mwh,
        case2_total_consumption_mwh,
        case3_total_consumption_mwh,
        case4_total_consumption_mwh
    ],
    'EV Consumption (MWh)': [
        case1_ev_mwh,
        case2_ev_mwh,
        case3_ev_mwh,
        case4_ev_mwh
    ],
    'HP Consumption (MWh)': [
        case1_hp_mwh,
        case2_hp_mwh,
        case3_hp_mwh,
        case4_hp_mwh
    ],
    'Inflex Load (MWh)': [
        case1_inflex_mwh,
        case2_inflex_mwh,
        case3_inflex_mwh,
        case4_inflex_mwh
    ],
    'PV Production (MWh)': [
        case1_pv_mwh,
        case2_pv_mwh,
        case3_pv_mwh,
        case4_pv_mwh
    ],
    'Grid Consumption (MWh)': [
        case1_grid_consumption_mwh,
        case2_grid_consumption_mwh,
        case3_grid_consumption_mwh,
        case4_grid_consumption_mwh
    ],
    'Grid Injection (MWh)': [
        case1_grid_injection_mwh,
        case2_grid_injection_mwh,
        case3_grid_injection_mwh,
        case4_grid_injection_mwh
    ],
    'Access Power (kW)': [
        case1_access_power_kw,
        case2_access_power_kw,
        case3_access_power_kw,
        case4_access_power_kw
    ],
    'Max Monthly Peak (kW)': [
        case1_max_peak_kw,
        case2_max_peak_kw,
        case3_max_peak_kw,
        case4_max_peak_kw
    ],
    'Total Cost (EUR)': [
        case1_total_cost,
        case2_total_cost,
        case3_total_cost,
        case4_total_cost
    ],
    # Detailed savings breakdown
    'Energy Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_energy_savings,
        case3_energy_savings,
        case4_energy_savings
    ],
    'Spot Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_spot_savings,
        case3_spot_savings,
        case4_spot_savings
    ],
    'Peak Savings Total (EUR)': [
        0.0,  # Case 1 is baseline
        case2_peak_savings_total,
        case3_peak_savings_total,
        case4_peak_savings_total
    ],
    'Access Power Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_access_savings,
        case3_access_savings,
        case4_access_savings
    ],
    'Monthly Peak Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_monthly_peak_savings,
        case3_monthly_peak_savings,
        case4_monthly_peak_savings
    ],
    'Exceedance Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_over_usage_savings,
        case3_over_usage_savings,
        case4_over_usage_savings
    ],
    'Total Savings (EUR)': [
        0.0,  # Case 1 is baseline
        case2_total_cost - case1_total_cost,
        case3_total_cost - case1_total_cost,
        case4_total_cost - case1_total_cost
    ],
    'Total Savings (%)': [
        0.0,  # Case 1 is baseline
        (case2_total_cost - case1_total_cost) / case1_total_cost * 100,
        (case3_total_cost - case1_total_cost) / case1_total_cost * 100,
        (case4_total_cost - case1_total_cost) / case1_total_cost * 100
    ]
})

print("="*80)
print("FOUR-CASE COMPARISON TABLE")
print("="*80)
display(comparison_table.style.format({
    'Total Consumption (MWh)': '{:,.2f}',
    'EV Consumption (MWh)': '{:,.2f}',
    'HP Consumption (MWh)': '{:,.2f}',
    'Inflex Load (MWh)': '{:,.2f}',
    'PV Production (MWh)': '{:,.2f}',
    'Grid Consumption (MWh)': '{:,.2f}',
    'Grid Injection (MWh)': '{:,.2f}',
    'Access Power (kW)': '{:,.2f}',
    'Max Monthly Peak (kW)': '{:,.2f}',
    'Total Cost (EUR)': '{:,.2f}',
    'Energy Savings (EUR)': '{:,.2f}',
    'Spot Savings (EUR)': '{:,.2f}',
    'Peak Savings Total (EUR)': '{:,.2f}',
    'Access Power Savings (EUR)': '{:,.2f}',
    'Monthly Peak Savings (EUR)': '{:,.2f}',
    'Exceedance Savings (EUR)': '{:,.2f}',
    'Total Savings (EUR)': '{:,.2f}',
    'Total Savings (%)': '{:,.2f}',
}))

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Case 1 (Baseline):                    {case1_total_cost:>15,.2f} EUR")
print(f"Case 2 (Optimized HP only):           {case2_total_cost:>15,.2f} EUR  (Savings: {case2_total_cost - case1_total_cost:>10,.2f} EUR)")
print(f"  - Energy Savings:                   {case2_energy_savings:>15,.2f} EUR")
print(f"  - Spot Savings:                      {case2_spot_savings:>15,.2f} EUR")
print(f"  - Peak Savings Total:                {case2_peak_savings_total:>15,.2f} EUR")
print(f"    * Access Power Savings:             {case2_access_savings:>15,.2f} EUR")
print(f"    * Monthly Peak Savings:             {case2_monthly_peak_savings:>15,.2f} EUR")
print(f"    * Exceedance Savings:               {case2_over_usage_savings:>15,.2f} EUR")
print(f"Case 3 (Optimized EV only):           {case3_total_cost:>15,.2f} EUR  (Savings: {case3_total_cost - case1_total_cost:>10,.2f} EUR)")
print(f"  - Energy Savings:                   {case3_energy_savings:>15,.2f} EUR")
print(f"  - Spot Savings:                      {case3_spot_savings:>15,.2f} EUR")
print(f"  - Peak Savings Total:                {case3_peak_savings_total:>15,.2f} EUR")
print(f"    * Access Power Savings:             {case3_access_savings:>15,.2f} EUR")
print(f"    * Monthly Peak Savings:             {case3_monthly_peak_savings:>15,.2f} EUR")
print(f"    * Exceedance Savings:               {case3_over_usage_savings:>15,.2f} EUR")
print(f"Case 4 (Optimized EV + HP):           {case4_total_cost:>15,.2f} EUR  (Savings: {case4_total_cost - case1_total_cost:>10,.2f} EUR)")
print(f"  - Energy Savings:                   {case4_energy_savings:>15,.2f} EUR")
print(f"  - Spot Savings:                      {case4_spot_savings:>15,.2f} EUR")
print(f"  - Peak Savings Total:                {case4_peak_savings_total:>15,.2f} EUR")
print(f"    * Access Power Savings:             {case4_access_savings:>15,.2f} EUR")
print(f"    * Monthly Peak Savings:             {case4_monthly_peak_savings:>15,.2f} EUR")
print(f"    * Exceedance Savings:               {case4_over_usage_savings:>15,.2f} EUR")
print("="*80)

### Visualize Four-Case Comparison

This section visualizes the comparison between all four cases, showing consumption breakdowns, peak powers, and cost savings.

In [ ]:
# Visualization: Consumption Breakdown
# Check if comparison_table exists (must run previous cell first)
if 'comparison_table' not in globals():
    raise NameError(
        "comparison_table is not defined. Please run the previous cell "
        "('Four-Case Comparison Table') first to create the comparison table."
    )

# Define case labels with abbreviations
case_labels = ['Case 1: UC EV + UC HP', 'Case 2: O HP + UC EV', 'Case 3: O EV + UC HP', 'Case 4: O EV + O HP']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(comparison_table))
width = 0.6

# Plot 1: Consumption breakdown (stacked bars)
cases = comparison_table['Case'].values
inflex_values = comparison_table['Inflex Load (MWh)'].values
ev_values = comparison_table['EV Consumption (MWh)'].values
hp_values = comparison_table['HP Consumption (MWh)'].values

ax1.bar(x, inflex_values, width, label='Inflex Load', color='gray', alpha=0.7)
ax1.bar(x, ev_values, width, bottom=inflex_values, label='EV Consumption', color='blue', alpha=0.7)
ax1.bar(x, hp_values, width, bottom=inflex_values + ev_values, label='HP Consumption', color='red', alpha=0.7)

ax1.set_xlabel('Case', fontsize=12)
ax1.set_ylabel('Consumption (MWh)', fontsize=12)
ax1.set_title('Total Consumption Breakdown by Case', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(case_labels, rotation=45, ha='right')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Total costs and savings
costs = comparison_table['Total Cost (EUR)'].values
colors_cost = ['red', 'orange', 'lightblue', 'green']
bars = ax2.bar(x, costs, width, color=colors_cost, alpha=0.7)

ax2.set_xlabel('Case', fontsize=12)
ax2.set_ylabel('Total Cost (EUR)', fontsize=12)
ax2.set_title('Total Cost Comparison', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(case_labels, rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, costs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualization: Peak Power and Access Power Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Max Monthly Peak
max_peaks = comparison_table['Max Monthly Peak (kW)'].values
bars1 = ax1.bar(x, max_peaks, width, color=colors_cost, alpha=0.7)
ax1.set_xlabel('Case', fontsize=12)
ax1.set_ylabel('Max Monthly Peak (kW)', fontsize=12)
ax1.set_title('Maximum Monthly Peak Power', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(case_labels, rotation=45, ha='right')
ax1.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars1, max_peaks):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 2: Access Power
access_powers = comparison_table['Access Power (kW)'].values
bars2 = ax2.bar(x, access_powers, width, color=colors_cost, alpha=0.7)
ax2.set_xlabel('Case', fontsize=12)
ax2.set_ylabel('Access Power (kW)', fontsize=12)
ax2.set_title('Access Power (Used for Billing)', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(case_labels, rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars2, access_powers):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualization: Savings vs Baseline (Stacked Bar Chart)
fig, ax = plt.subplots(figsize=(12, 6))

# Get savings components from comparison table (skip Case 1 which has 0 savings)
energy_savings = comparison_table['Energy Savings (EUR)'].values[1:]  # Cases 2, 3, 4
spot_savings = comparison_table['Spot Savings (EUR)'].values[1:]
access_savings = comparison_table['Access Power Savings (EUR)'].values[1:]
monthly_peak_savings = comparison_table['Monthly Peak Savings (EUR)'].values[1:]
exceedance_savings = comparison_table['Exceedance Savings (EUR)'].values[1:]

# Create x positions for cases 2, 3, 4 (skip Case 1)
x_savings = np.arange(3)  # Only 3 cases to show (2, 3, 4)

# Stack bars: energy (bottom), then spot, then access power, then monthly peak, then exceedance
ax.bar(x_savings, energy_savings, width, label='Energy Savings', color='#2E86AB', alpha=0.8)
ax.bar(x_savings, spot_savings, width, bottom=energy_savings, label='Spot Savings', color='#A23B72', alpha=0.8)
ax.bar(x_savings, access_savings, width, 
       bottom=energy_savings + spot_savings, 
       label='Access Power Savings', color='#F18F01', alpha=0.8)
ax.bar(x_savings, monthly_peak_savings, width, 
       bottom=energy_savings + spot_savings + access_savings, 
       label='Monthly Peak Savings', color='#C73E1D', alpha=0.8)
ax.bar(x_savings, exceedance_savings, width, 
       bottom=energy_savings + spot_savings + access_savings + monthly_peak_savings, 
       label='Exceedance Savings', color='#6A994E', alpha=0.8)

ax.set_xlabel('Case', fontsize=12)
ax.set_ylabel('Total Savings vs Case 1 (EUR)', fontsize=12)
ax.set_title('Cost Savings Compared to Baseline (Case 1: UC EV + UC HP)', fontsize=14, fontweight='bold')
ax.set_xticks(x_savings)
ax.set_xticklabels(case_labels[1:], rotation=45, ha='right')  # Skip Case 1 label
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Add total savings labels on top of bars
total_savings = comparison_table['Total Savings (EUR)'].values[1:]
for i, (x_pos, total) in enumerate(zip(x_savings, total_savings)):
    ax.text(x_pos, total,
            f'{total:,.0f}', ha='center', va='bottom' if total > 0 else 'top', 
            fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### Monthly Savings Breakdown by Case

This section shows monthly savings breakdown for each optimized case (Case 2, 3, and 4) compared to the baseline (Case 1). Each plot shows stacked bars representing:
- **Energy Savings**: Difference in energy-based costs
- **Spot Savings**: Difference in spot price costs
- **Access Power Savings**: Difference in access power costs
- **Monthly Peak Savings**: Difference in monthly peak costs

In [ ]:
# Calculate monthly savings for each case vs Case 1 (baseline)
# Ensure all billing dataframes are available
if 'baseline_hp_bills' not in globals() or 'case2_bills' not in globals() or 'case3_bills' not in globals() or 'optimized_bills' not in globals():
    raise NameError("Billing dataframes not found. Please run the previous cells to calculate all cases first.")

# Prepare monthly savings data
months = baseline_hp_bills['month'].values

# Case 2 monthly savings vs Case 1
case2_monthly_energy_savings = case2_bills['energy_cost_eur'].values - baseline_hp_bills['energy_cost_eur'].values
case2_monthly_spot_savings = case2_bills['spot_cost_eur'].values - baseline_hp_bills['spot_cost_eur'].values
case2_monthly_access_savings = case2_bills['access_cost_eur'].values - baseline_hp_bills['access_cost_eur'].values
case2_monthly_peak_savings = case2_bills['monthly_peak_cost_eur'].values - baseline_hp_bills['monthly_peak_cost_eur'].values

# Case 3 monthly savings vs Case 1
case3_monthly_energy_savings = case3_bills['energy_cost_eur'].values - baseline_hp_bills['energy_cost_eur'].values
case3_monthly_spot_savings = case3_bills['spot_cost_eur'].values - baseline_hp_bills['spot_cost_eur'].values
case3_monthly_access_savings = case3_bills['access_cost_eur'].values - baseline_hp_bills['access_cost_eur'].values
case3_monthly_peak_savings = case3_bills['monthly_peak_cost_eur'].values - baseline_hp_bills['monthly_peak_cost_eur'].values

# Case 4 monthly savings vs Case 1
case4_monthly_energy_savings = optimized_bills['energy_cost_eur'].values - baseline_hp_bills['energy_cost_eur'].values
case4_monthly_spot_savings = optimized_bills['spot_cost_eur'].values - baseline_hp_bills['spot_cost_eur'].values
case4_monthly_access_savings = optimized_bills['access_cost_eur'].values - baseline_hp_bills['access_cost_eur'].values
case4_monthly_peak_savings = optimized_bills['monthly_peak_cost_eur'].values - baseline_hp_bills['monthly_peak_cost_eur'].values

# Create stacked bar charts for each case
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Case 2: Optimized HP + Uncontrolled EV
ax1 = axes[0]
x_pos = np.arange(len(months))
width = 0.6

# Stack bars: energy (bottom), then spot, then access power, then monthly peak
ax1.bar(x_pos, case2_monthly_energy_savings, width, label='Energy Savings', color='#2E86AB', alpha=0.8)
ax1.bar(x_pos, case2_monthly_spot_savings, width, bottom=case2_monthly_energy_savings, label='Spot Savings', color='#A23B72', alpha=0.8)
ax1.bar(x_pos, case2_monthly_access_savings, width, 
        bottom=case2_monthly_energy_savings + case2_monthly_spot_savings, 
        label='Access Power Savings', color='#F18F01', alpha=0.8)
ax1.bar(x_pos, case2_monthly_peak_savings, width, 
        bottom=case2_monthly_energy_savings + case2_monthly_spot_savings + case2_monthly_access_savings, 
        label='Monthly Peak Savings', color='#C73E1D', alpha=0.8)

ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Monthly Savings vs Case 1 (EUR)', fontsize=12)
ax1.set_title('Case 2: O HP + UC EV - Monthly Savings Breakdown', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(months, rotation=45, ha='right')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Case 3: Optimized EV + Uncontrolled HP
ax2 = axes[1]

ax2.bar(x_pos, case3_monthly_energy_savings, width, label='Energy Savings', color='#2E86AB', alpha=0.8)
ax2.bar(x_pos, case3_monthly_spot_savings, width, bottom=case3_monthly_energy_savings, label='Spot Savings', color='#A23B72', alpha=0.8)
ax2.bar(x_pos, case3_monthly_access_savings, width, 
        bottom=case3_monthly_energy_savings + case3_monthly_spot_savings, 
        label='Access Power Savings', color='#F18F01', alpha=0.8)
ax2.bar(x_pos, case3_monthly_peak_savings, width, 
        bottom=case3_monthly_energy_savings + case3_monthly_spot_savings + case3_monthly_access_savings, 
        label='Monthly Peak Savings', color='#C73E1D', alpha=0.8)

ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Monthly Savings vs Case 1 (EUR)', fontsize=12)
ax2.set_title('Case 3: O EV + UC HP - Monthly Savings Breakdown', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(months, rotation=45, ha='right')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Case 4: Optimized EV + Optimized HP
ax3 = axes[2]

ax3.bar(x_pos, case4_monthly_energy_savings, width, label='Energy Savings', color='#2E86AB', alpha=0.8)
ax3.bar(x_pos, case4_monthly_spot_savings, width, bottom=case4_monthly_energy_savings, label='Spot Savings', color='#A23B72', alpha=0.8)
ax3.bar(x_pos, case4_monthly_access_savings, width, 
        bottom=case4_monthly_energy_savings + case4_monthly_spot_savings, 
        label='Access Power Savings', color='#F18F01', alpha=0.8)
ax3.bar(x_pos, case4_monthly_peak_savings, width, 
        bottom=case4_monthly_energy_savings + case4_monthly_spot_savings + case4_monthly_access_savings, 
        label='Monthly Peak Savings', color='#C73E1D', alpha=0.8)

ax3.set_xlabel('Month', fontsize=12)
ax3.set_ylabel('Monthly Savings vs Case 1 (EUR)', fontsize=12)
ax3.set_title('Case 4: O EV + O HP - Monthly Savings Breakdown', fontsize=14, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(months, rotation=45, ha='right')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3, axis='y')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.show()

# Print summary statistics
print("="*80)
print("MONTHLY SAVINGS SUMMARY")
print("="*80)
print(f"\nCase 2: O HP + UC EV")
print(f"  Total Energy Savings:     {case2_monthly_energy_savings.sum():>15,.2f} EUR")
print(f"  Total Spot Savings:       {case2_monthly_spot_savings.sum():>15,.2f} EUR")
print(f"  Total Access Power Savings: {case2_monthly_access_savings.sum():>15,.2f} EUR")
print(f"  Total Monthly Peak Savings: {case2_monthly_peak_savings.sum():>15,.2f} EUR")
print(f"  Total Savings:            {(case2_monthly_energy_savings + case2_monthly_spot_savings + case2_monthly_access_savings + case2_monthly_peak_savings).sum():>15,.2f} EUR")

print(f"\nCase 3: O EV + UC HP")
print(f"  Total Energy Savings:     {case3_monthly_energy_savings.sum():>15,.2f} EUR")
print(f"  Total Spot Savings:       {case3_monthly_spot_savings.sum():>15,.2f} EUR")
print(f"  Total Access Power Savings: {case3_monthly_access_savings.sum():>15,.2f} EUR")
print(f"  Total Monthly Peak Savings: {case3_monthly_peak_savings.sum():>15,.2f} EUR")
print(f"  Total Savings:            {(case3_monthly_energy_savings + case3_monthly_spot_savings + case3_monthly_access_savings + case3_monthly_peak_savings).sum():>15,.2f} EUR")

print(f"\nCase 4: O EV + O HP")
print(f"  Total Energy Savings:     {case4_monthly_energy_savings.sum():>15,.2f} EUR")
print(f"  Total Spot Savings:       {case4_monthly_spot_savings.sum():>15,.2f} EUR")
print(f"  Total Access Power Savings: {case4_monthly_access_savings.sum():>15,.2f} EUR")
print(f"  Total Monthly Peak Savings: {case4_monthly_peak_savings.sum():>15,.2f} EUR")
print(f"  Total Savings:            {(case4_monthly_energy_savings + case4_monthly_spot_savings + case4_monthly_access_savings + case4_monthly_peak_savings).sum():>15,.2f} EUR")
print("="*80)

## Export results for downstream notebooks

Writes CSVs to `output/notebooks/` for reuse in online MPC notebooks (e.g. notebook 11), mirroring notebook 03:

- **15-min** full optimizer trace (`results_df`, incl. `access_power`, grid, EV, HP, buffer SOC)
- **Monthly bills** (optimized joint scenario)
- **Monthly injection** (optimized)
- **Monthly comparison** (baseline vs optimized, incl. access power and cost deltas)

Run after the optimization and billing cells above (`results_df`, `optimized_bills`, `optimized_injection`, `monthly_comparison`).

In [ ]:
# Export deterministic EV+HP MPC results for reuse in online notebooks (e.g. Notebook 11)

from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
NOTEBOOKS_OUTPUT_DIR = PROJECT_ROOT / "output" / "notebooks"
NOTEBOOKS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_required = [
    "results_df",
    "optimized_bills",
    "optimized_injection",
    "baseline_hp_bills",
    "baseline_hp_injection",
    "monthly_comparison",
]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise NameError(
        f"Missing objects {_missing}. Run optimization, billing, and monthly comparison cells first."
    )

# 15-min deterministic results (full year)
export_det_15min = results_df.copy()
export_det_15min["timestamp"] = pd.to_datetime(export_det_15min["timestamp"], errors="coerce")
export_det_15min_path = NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_15min_notebook_04.csv"
export_det_15min.to_csv(export_det_15min_path, index=False)

# Monthly deterministic bills — optimized (offtake)
export_det_monthly_bills = optimized_bills.copy()
export_det_monthly_bills_path = (
    NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_monthly_bills_notebook_04.csv"
)
export_det_monthly_bills.to_csv(export_det_monthly_bills_path, index=False)

# Monthly deterministic injection — optimized
export_det_monthly_inj = optimized_injection.copy()
export_det_monthly_inj_path = (
    NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_monthly_injection_notebook_04.csv"
)
export_det_monthly_inj.to_csv(export_det_monthly_inj_path, index=False)

# Monthly baseline bills (uncontrolled EV + uncontrolled HP, heuristic access)
export_baseline_monthly_bills = baseline_hp_bills.copy()
export_baseline_monthly_bills_path = (
    NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_monthly_baseline_bills_notebook_04.csv"
)
export_baseline_monthly_bills.to_csv(export_baseline_monthly_bills_path, index=False)

# Baseline injection
export_baseline_monthly_inj = baseline_hp_injection.copy()
export_baseline_monthly_inj_path = (
    NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_monthly_baseline_injection_notebook_04.csv"
)
export_baseline_monthly_inj.to_csv(export_baseline_monthly_inj_path, index=False)

# Baseline vs optimized monthly summary (access power, peaks, costs)
export_monthly_comparison = monthly_comparison.copy()
export_monthly_comparison_path = (
    NOTEBOOKS_OUTPUT_DIR / "deterministic_ev_hp_monthly_comparison_notebook_04.csv"
)
export_monthly_comparison.to_csv(export_monthly_comparison_path, index=False)

# Annual net costs for quick checks (same definition as billing cells above)
_det_inj_rev = optimized_injection["injection_net_revenue_eur"].sum()
_det_net = optimized_bills["total_cost_eur"].sum() - _det_inj_rev
_base_inj_rev = baseline_hp_injection["injection_net_revenue_eur"].sum()
_base_net = baseline_hp_bills["total_cost_eur"].sum() - _base_inj_rev

print("Exported deterministic EV+HP MPC results:")
print(f"- 15-min results:              {export_det_15min_path}")
print(f"- Monthly bills (optimized):   {export_det_monthly_bills_path}")
print(f"- Monthly injection (optimized): {export_det_monthly_inj_path}")
print(f"- Monthly bills (baseline):      {export_baseline_monthly_bills_path}")
print(f"- Monthly injection (baseline): {export_baseline_monthly_inj_path}")
print(f"- Monthly comparison:          {export_monthly_comparison_path}")
print(f"\nAnnual net cost baseline:      {_base_net:,.2f} EUR")
print(f"Annual net cost deterministic: {_det_net:,.2f} EUR")
print(f"Annual savings:                {_base_net - _det_net:,.2f} EUR")